# Feature Ideas

- Studies features from the table but make it into a number:
    - `study_type` (make it categorical)
    - `target_duration` make it into number of years
    - `phase` (make it categorical)
    - `enrollment` 
    - `number_of_arms`
    - `number_of_groups`
    - `has_dmc`
    - `is_fda_regulated_drug`
    - `is_fda_regulated_device`
    - `is_unapproved_device`
    - `is_ppsd`
    - `is_us_export`
    - `biospec_retention`
    - `plan_to_share_ipd`

- designs table
    - `allocation`
    - `intervention_model`
    - `observational_model`
    - `time_perspective`
    - `masking`
    - `subject_masked`	
    - `caregiver_masked`
    - `investigator_masked`
    - `outcomes_assessor_masked`

- eligibilities table
    - `sampling_method`
    - `gender`
    - `minimum_age`
    - `maximum_age`
    - `healthy_volunteers`	`
    - `adult`	
    - `child`
    - `older_adult`

- sponsors
    - `agency_class`

- Derived features:
    - `age_range`: `maximum_age` - `minimum_age`
    - `num_outcomes`: number of outcomes measured
    - `num_of_words_trial_description`
    - `num_of_words_trial_title`
    - `num_of_words_eligibility_criteria`

    - `num_intervenctions`: number of interventions for this trial (more drugs may correspond to cocktail or non-inferiority trial)
    - `num_conditions`: number of conditions for this trial (a trial could target lots of diseases)
    - `num_sponsors`: number of sponsors for this trial (more sponsors may lead to higher success rate?)
    - `num_facilities`: number of facilities for this trial (more sites have more confidence)

    - `num_history_studies_intervention`: number of historical trials for the intervention in this trial (measuring if this intervention drug is a novel drug or not)
    - `num_history_studies_condition`: number of historical trials for the condition in this trial (measuring if this condition is well studied)
    - `num_history_studies_sponsor`: number of historical trials that this sponsor have (measuring experience of the sponsor)
    - `num_history_studies_facility`: number of historical trials that this facility have (measuring experience of the facility)

    - `in_us`: if the trial facility locates in US

    - `avg_condition_historical_success_rate`: for this condition, what is historical success rate? 
    - `avg_facility_historical_success_rate`: for this facility, what is historical success rate? 
    - `avg_sponsor_historical_success_rate`: for this sponsor, what is historical success rate? 
    - `avg_intervention_historical_success_rate`: for this intervention, what is historical success rate? 

    - `time_since_last_trial_sponsor`
    - `time_since_last_trial_facility`


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from relbench.datasets import get_dataset
from relbench.tasks import get_task
from tqdm import tqdm 
from pandarallel import pandarallel

%load_ext autoreload
%autoreload 2


In [ ]:
dataset = get_dataset(name='rel-trial', process=False)
task = get_task('rel-trial', 'study-outcome', download=True)

In [ ]:
from tqdm import tqdm 
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False, nb_workers = 10)

def get_success_rate(transaction, column, nct_id, timestamp, outcome_analyses):
    nct_history = transaction[(transaction[column].isin(transaction[transaction.nct_id == nct_id][column].values)) & (transaction.date < timestamp)].nct_id.unique()
    outcome_temp = outcome_analyses[outcome_analyses.nct_id.isin(nct_history)]
    outcome_temp = outcome_temp[(outcome_temp.p_value_modifier.isnull()) | (outcome_temp.p_value_modifier != '>')]
    outcome_temp = outcome_temp[(outcome_temp.p_value >= 0) & (outcome_temp.p_value <= 1) & (outcome_temp.outcome_type == 'Primary')]
    nct2p = outcome_temp.groupby('nct_id').p_value.min()
    if len(nct2p) == 0:
        return 0
    else:
        return sum(nct2p<= 0.05)/len(nct2p)

def get_features(train_df):
    dataset = get_dataset(name='rel-trial', process=False)
    task = get_task('rel-trial', 'study-outcome', download=True)
    tables = dataset.get_db().table_dict

    interventions_studies = tables['interventions_studies'].df
    conditions_studies = tables['conditions_studies'].df
    sponsors_studies = tables['sponsors_studies'].df
    facilities_studies = tables['facilities_studies'].df
    studies = tables['studies'].df
    designs = tables['designs'].df
    eligibilities = tables['eligibilities'].df
    sponsors = tables['sponsors'].df

    outcomes = tables['outcomes'].df
    facilities = tables['facilities'].df
    outcome_analyses = tables['outcome_analyses'].df
    outcome_analyses = outcome_analyses.merge(outcomes[['id', 'outcome_type']], left_on='outcome_id', right_on = 'id')

    eligibilities['minimum_age'] = eligibilities['minimum_age'].apply(lambda x: int(x.split('Years')[0]) if (x is not None) and ('Years' in x) else -1)
    eligibilities['maximum_age'] = eligibilities['maximum_age'].apply(lambda x: int(x.split('Years')[0]) if (x is not None) and ('Years' in x) else -1)

    train_studies = train_df.merge(studies)

    train_studies['is_observational'] = train_studies.study_type.apply(lambda x: 1 if x in ['Observational', 'Observational [Patient Registry]'] else 0)
    train_studies['is_interventional'] = train_studies.study_type.apply(lambda x: 1 if x in ['Interventional'] else 0)
    train_studies['is_expanded_access'] = train_studies.study_type.apply(lambda x: 1 if x in ['Expanded Access'] else 0)

    train_studies['is_phase_1'] = train_studies.phase.apply(lambda x: 1 if (x is not None) and ('Phase 1' in x) else 0)
    train_studies['is_phase_2'] = train_studies.phase.apply(lambda x: 1 if (x is not None) and ('Phase 2' in x) else 0)
    train_studies['is_phase_3'] = train_studies.phase.apply(lambda x: 1 if (x is not None) and ('Phase 3' in x) else 0)
    train_studies['is_phase_4'] = train_studies.phase.apply(lambda x: 1 if (x is not None) and ('Phase 4' in x) else 0)

    train_studies = train_studies[['nct_id', 'is_observational', 'is_interventional', 'is_expanded_access', 'is_phase_1', 'is_phase_2', 'is_phase_3', 'is_phase_4', 
                'enrollment', 'number_of_arms', 'number_of_groups', 'has_dmc', 'is_fda_regulated_drug', 'is_fda_regulated_device', 'is_unapproved_device', 
                'is_ppsd', 'is_us_export', 'biospec_retention', 'plan_to_share_ipd']]

    train_df = train_df.merge(train_studies)

    train_design = train_df.merge(designs)
    train_design['is_randomized'] = train_design.allocation.apply(lambda x: 1 if x == 'Randomized' else 0)
    train_design['is_parallel_assignment'] = train_design.intervention_model.apply(lambda x: 1 if x == 'Parallel Assignment' else 0)
    train_design['is_single_group_assignment'] = train_design.intervention_model.apply(lambda x: 1 if x == 'Single Group Assignment' else 0)
    train_design['is_crossover_assignment'] = train_design.intervention_model.apply(lambda x: 1 if x == 'Crossover Assignment' else 0)
    train_design['is_factorial_assignment'] = train_design.intervention_model.apply(lambda x: 1 if x == 'Factorial Assignment' else 0)
    train_design['is_sequential_assignment'] = train_design.intervention_model.apply(lambda x: 1 if x == 'Sequential Assignment' else 0)

    train_design['is_single_masking'] = train_design.masking.apply(lambda x: 1 if x == 'Single' else 0)
    train_design['is_double_masking'] = train_design.masking.apply(lambda x: 1 if x == 'Double' else 0)
    train_design['is_triple_masking'] = train_design.masking.apply(lambda x: 1 if x == 'Triple' else 0)
    train_design['is_quadruple_masking'] = train_design.masking.apply(lambda x: 1 if x == 'Quadruple' else 0)
    train_design['is_no_masking'] = train_design.masking.apply(lambda x: 1 if x == 'None (Open Label)' else 0)

    train_design = train_design[['nct_id', 'is_randomized', 'is_parallel_assignment', 'is_single_group_assignment', 
    'is_crossover_assignment', 'is_factorial_assignment', 'is_sequential_assignment',
    'is_single_masking', 'is_double_masking', 'is_triple_masking', 'is_quadruple_masking', 'is_no_masking', 'primary_purpose', 
    'subject_masked', 'caregiver_masked', 'investigator_masked', 'outcomes_assessor_masked']]

    train_df = train_df.merge(train_design)

    train_eligibility = train_df.merge(eligibilities)
    train_eligibility['is_non_probability_sample'] = train_eligibility['sampling_method'].apply(lambda x: 1 if x == 'Non-Probability Sample' else 0)
    train_eligibility['is_female_only'] = train_eligibility['gender'].apply(lambda x: 1 if x == 'Female' else 0)
    train_eligibility['is_male_only'] = train_eligibility['gender'].apply(lambda x: 1 if x == 'Male' else 0)
    train_eligibility['accept_healthy_volunteer'] = train_eligibility['healthy_volunteers'].apply(lambda x: 1 if x == 'Accepts Healthy Volunteers' else 0)
    train_eligibility['is_min_age_ge_60'] = train_eligibility['minimum_age'].apply(lambda x: 1 if x >= 60 else 0)
    train_eligibility['is_max_age_le_20'] = train_eligibility['maximum_age'].apply(lambda x: 1 if x <= 20 else 0)
    train_eligibility = train_eligibility[['nct_id', 'is_non_probability_sample', 'is_female_only', 'is_male_only', 'accept_healthy_volunteer', 'is_min_age_ge_60']]

    train_df = train_df.merge(train_eligibility, how = 'left')

    train_df['num_history_intervention'] = train_df.parallel_apply(lambda x: interventions_studies[(interventions_studies.intervention_id.isin(interventions_studies[interventions_studies.nct_id == x.nct_id].intervention_id.values)) & (interventions_studies.date < x.timestamp)].groupby('intervention_id').nct_id.agg(len).mean(), axis = 1)
    train_df['num_history_condition'] = train_df.parallel_apply(lambda x: conditions_studies[(conditions_studies.condition_id.isin(conditions_studies[conditions_studies.nct_id == x.nct_id].condition_id.values)) & (conditions_studies.date < x.timestamp)].groupby('condition_id').nct_id.agg(len).mean(), axis = 1)
    train_df['num_history_sponsor'] = train_df.parallel_apply(lambda x: sponsors_studies[(sponsors_studies.sponsor_id.isin(sponsors_studies[sponsors_studies.nct_id == x.nct_id].sponsor_id.values)) & (sponsors_studies.date < x.timestamp)].groupby('sponsor_id').nct_id.agg(len).mean(), axis = 1)
    train_df['num_history_facility'] = train_df.parallel_apply(lambda x: facilities_studies[(facilities_studies.facility_id.isin(facilities_studies[facilities_studies.nct_id == x.nct_id].facility_id.values)) & (facilities_studies.date < x.timestamp)].groupby('facility_id').nct_id.agg(len).mean(), axis = 1)
    train_df.fillna(0, inplace=True)

    # Calculate derived features for temporal_studies
    eligibilities = tables['eligibilities'].df
    studies = tables['studies'].df
    outcomes = tables['outcomes'].df
    facilities = tables['facilities'].df
    outcome_analyses = tables['outcome_analyses'].df
    outcome_analyses = outcome_analyses.merge(outcomes[['id', 'outcome_type']], left_on='outcome_id', right_on = 'id')

    # Age range
    eligibilities['age_range'] = eligibilities['maximum_age'] - eligibilities['minimum_age']

    # Number of words in eligibility criteria
    eligibilities['num_of_words_eligibility_criteria'] = eligibilities['criteria'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

    # Number of words in trial title and description
    studies['num_of_words_trial_title'] = studies['brief_title'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)
    studies['num_of_words_trial_description'] = studies['official_title'].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

    # Number of outcomes
    outcomes_counts = outcomes.groupby('nct_id').size().reset_index(name='num_outcomes')

    # In US
    facilities['in_us'] = facilities['country'].apply(lambda x: 1 if x in ['US', 'United States'] else 0)
    nct_id2in_us = facilities.merge(facilities_studies, on='facility_id').groupby('nct_id').in_us.max().reset_index()

    train_df['history_success_rate_facility'] = train_df.parallel_apply(lambda x: get_success_rate(facilities_studies, 'facility_id', x.nct_id, x.timestamp, outcome_analyses), axis = 1)
    train_df['history_success_rate_sponsor'] = train_df.parallel_apply(lambda x: get_success_rate(sponsors_studies, 'sponsor_id', x.nct_id, x.timestamp, outcome_analyses), axis = 1)
    train_df['history_success_rate_condition'] = train_df.parallel_apply(lambda x: get_success_rate(conditions_studies, 'condition_id', x.nct_id, x.timestamp, outcome_analyses), axis = 1)
    train_df['history_success_rate_intervention'] = train_df.parallel_apply(lambda x: get_success_rate(interventions_studies, 'intervention_id', x.nct_id, x.timestamp, outcome_analyses), axis = 1)

    # Merge derived features
    derived_features = train_df

    derived_features = derived_features.merge(eligibilities[['nct_id', 'age_range', 'num_of_words_eligibility_criteria']], on='nct_id', how='left')
    derived_features = derived_features.merge(studies[['nct_id', 'num_of_words_trial_title', 'num_of_words_trial_description']], on='nct_id', how='left')
    derived_features = derived_features.merge(outcomes_counts, on='nct_id', how='left')
    derived_features.fillna(0, inplace=True)

    # Check if facilities are in US
    derived_features = derived_features.merge(nct_id2in_us, how = 'left')
    return derived_features

train_df = task.get_table("train").df

In [ ]:
train_df = get_features(train_df)

In [ ]:
val_df = task.get_table("val").df
val_df = get_features(val_df)

In [ ]:
train_df.iloc[0]

In [ ]:
from torch_frame import TaskType, stype
from torch_frame.gbdt import LightGBM, XGBoost
from torch_frame.data import Dataset
from torch_frame.typing import Metric
from torch_frame.utils import infer_df_stype

In [ ]:
col_to_stype = infer_df_stype(train_df)
col_to_stype

In [ ]:
DROP_COLS = [
    # drop identifier cols
    'nct_id',
    'timestamp',
    #'history_success_rate_facility',
    #'history_success_rate_sponsor', 
    #'history_success_rate_condition',
    #'history_success_rate_intervention'
]
for c in DROP_COLS:
    del col_to_stype[c]

train_dset = Dataset(
    train_df.drop(DROP_COLS, axis=1),
    col_to_stype=col_to_stype,
    target_col='outcome'
).materialize()

val_tf = train_dset.convert_to_tensor_frame(val_df.drop(DROP_COLS, axis=1))
tune_metric = Metric.ROCAUC
print(train_dset.tensor_frame.num_cols, train_dset.tensor_frame.num_rows)

In [ ]:
gbdt = LightGBM(TaskType.BINARY_CLASSIFICATION, metric=tune_metric)
gbdt.tune(tf_train=train_dset.tensor_frame, tf_val=val_tf, num_trials=1)

In [ ]:
gbdt.save('data/outcome_xgb_no_history.json')

# Val Eval

In [ ]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, average_precision_score, accuracy_score, PrecisionRecallDisplay

In [ ]:
from sklearn.feature_selection import mutual_info_classif

def feature_summary_df(df: pd.DataFrame, y_col: str) -> pd.DataFrame:
    y = df[y_col]
    df = df.drop(y_col, axis=1)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return pd.DataFrame(
        {
            'Label Corr.': df.corrwith(y),
            'Label MI': mutual_info_classif(df.fillna(-1).values, y),
            'NaN %': df.isna().mean(),
        },
        index=df.columns
    ).sort_values(by='Label Corr.', ascending=False)


In [ ]:
(
    feature_summary_df(val_df.drop(DROP_COLS, axis=1), 'outcome')
    .style
    .format('{:.3f}', subset=['Label Corr.', 'Label MI'])
    .format('{:.1%}', subset=['NaN %'])
    .background_gradient(subset=['Label MI'])
)

In [ ]:
pred = gbdt.predict(tf_test=val_tf)

In [ ]:
auc = gbdt.compute_metric(val_tf.y, pred)
y_ = val_tf.y.numpy()
pred_ = pred.numpy()
ap = average_precision_score(y_, pred_)
acc = accuracy_score(y_, (pred_ > 0.5))

print(f"AUC: {auc:.3f}")
print(f"AP: {ap:.3f}")
print(f"Accuracy: {acc:.3f}")

In [ ]:
fig = go.Figure()
fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)
fpr, tpr, thresholds = roc_curve(y_, pred_)
fig.add_trace(
    go.Scatter(x=fpr, y=tpr, mode='lines')
)
fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain='domain'),
    width=600, height=600
)


In [ ]:
display = PrecisionRecallDisplay.from_predictions(
    y_, pred_, name='XGB', plot_chance_level=True
)
_ = display.ax_.set_title("2-class Precision-Recall curve")

In [ ]:
import shap

shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[:10_000])
shap_values = explainer.shap_values(val_arr, pred_[:10_000])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=55,
                  feature_names=feat_names)

In [ ]:
import shap

shap.initjs()
explainer = shap.TreeExplainer(gbdt.model)

val_arr, _, _ = gbdt._to_lightgbm_input(val_tf[:10_000])
shap_values = explainer.shap_values(val_arr, pred_[:10_000])

# TODO verify
feat_names = val_tf.col_names_dict.get(stype.categorical, []) + val_tf.col_names_dict[stype.numerical]

shap.summary_plot(shap_values, val_arr, plot_type='violin', max_display=55,
                  feature_names=feat_names)

In [ ]:
test_df = task.get_table("test").df
test_df = get_features(test_df)
test_tf = train_dset.convert_to_tensor_frame(test_df.drop(DROP_COLS, axis=1))
pred = gbdt.predict(tf_test=test_tf)
task.evaluate(pred)

In [ ]:
test_df_train_time_frame = test_df

In [ ]:
test_df

In [ ]:
val_pred = gbdt.predict(val_tf)

In [ ]:
task.evaluate(val_pred, task.get_table("val"))

In [ ]:
train_df['history_success_rate_facility'] = train_df.parallel_apply(lambda x: get_success_rate(facilities_studies, 'facility_id', x.nct_id, x.timestamp, outcome_analyses), axis = 1)


In [ ]:

tables = dataset.get_db().table_dict

interventions_studies = tables['interventions_studies'].df
conditions_studies = tables['conditions_studies'].df
sponsors_studies = tables['sponsors_studies'].df
facilities_studies = tables['facilities_studies'].df

In [ ]:
train_df.timestamp.iloc[0]

In [ ]:
facilities_studies[facilities_studies.date < train_df.timestamp.iloc[0]].date.max()

In [ ]:
train_df.timestamp.iloc[0]

In [ ]:
task_name = 'rel-trial-outcome'
train_df.to_csv(task_name + '-train.csv', index = False)
val_df.to_csv(task_name + '-val.csv', index = False)
test_df.to_csv(task_name + '-test.csv', index = False)

In [ ]:
test_df.timestamp.max()

In [ ]:
val_df